In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

In [27]:
# SPECTROMETER_FREQUENCY_MHZ = 600.0
MNOVA_PEAKS_FILES = [
    "../data/eksperyment2_bis-click/peaks_from_mnova/poReakcji_bis-click.csv",
    "../data/eksperyment2_bis-click/peaks_from_mnova/substraty_bis-click.csv"
]

OUTPUT_DIR = Path("../data/eksperyment2_bis-click/processed_peaks_lists")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# peaks squizeeing parameters (omit large gaps between peaks)
POSITION_START = 0.0
MAXIMAL_GAP_HZ = 20.0

SQUEEZE_PEAKS = True
MAX_WIDTH_HZ = 3.1

In [28]:
def parse_file(file_path, separator=","):
    lines = Path(file_path).read_text().splitlines()[1:] # skip header

    peaks_data = []
    for line in lines:
        line_parts = [part for part in line.split(separator) if part]
        peaks_data.extend([
            {
                "multiplet_name": line_parts[1],
                "peak_in_multiplet": int(line_parts[i]),
                "position_ppm": float(line_parts[i + 1]),
                "position_hz": float(line_parts[i + 2]),
                "height": float(line_parts[i + 3]),
                "width_hz": float(line_parts[i + 4]),
                "L/G": float(line_parts[i + 5]),
                "area": float(line_parts[i + 6]),
                }
            for i in range(5, len(line_parts), 7)
        ])        
    return peaks_data

def squeeze_peaks_data(df, position_start, maximal_gap_hz):
    df = df.copy().sort_values("position_hz").reset_index(drop=True)
    df.reset_index(drop=True, inplace=True) #.
    df["position_hz"] -= df["position_hz"].min() - POSITION_START
    for i in range(1, len(df)):
        gap = df["position_hz"].iloc[i] - df["position_hz"].iloc[i - 1]
        if gap > MAXIMAL_GAP_HZ:
            shift = gap - MAXIMAL_GAP_HZ
            df.loc[i:, "position_hz"] -= shift # we can use .loc (not .iloc) because we reset the index earlier
    return df

In [29]:
filtering_string = f"filtered{MAX_WIDTH_HZ:.1f}Hz" if MAX_WIDTH_HZ is not None else "unfiltered"
squeeze_string = "squeezed" if SQUEEZE_PEAKS else "unsqueezed"

for csv_file_path in MNOVA_PEAKS_FILES:
    peaks_data = parse_file(csv_file_path)
    output_file_path = OUTPUT_DIR / (Path(csv_file_path).stem + f"_{filtering_string}_{squeeze_string}.csv")
    peaks_data = pd.DataFrame(peaks_data)
    # peaks_data["position_hz_byhand"] = peaks_data["position_ppm"] * SPECTROMETER_FREQUENCY_MHZ
    peaks_data["gaussian_fraction"] = np.clip(1 / (1 + np.clip(peaks_data["L/G"], 0, None)), 0.0, 1.0)
    peaks_data = peaks_data.drop(columns=["L/G"])
    if SQUEEZE_PEAKS:
        squeezed_data = squeeze_peaks_data(peaks_data, POSITION_START, MAXIMAL_GAP_HZ)
    else:
        squeezed_data = peaks_data
    if MAX_WIDTH_HZ is not None:
        squeezed_data = squeezed_data.loc[squeezed_data["width_hz"] <= MAX_WIDTH_HZ]
    squeezed_data.to_csv(output_file_path, index=False)
    

## Sprawdzenie

In [22]:
run_filtered = pd.read_csv("../data/multiplets_lists/substraty_mieszanina.csv")
mieszanina_filtered = pd.read_csv("../data/multiplets_lists/mieszanina_po_reakcji_2-bezszerokich_squeezed-0.0-20.0Hz.csv")
mieszanina_all = pd.read_csv("../data/multiplets_lists/mieszanina_po_reakcji_2_squeezed-0.0-20.0Hz.csv")
substrat1_all = pd.read_csv("../data/multiplets_lists/azydekbenzylu_sub1_mono-click.csv")
substrat2_all = pd.read_csv("../data/multiplets_lists/fenyloacetylen_sub2_mono-click.csv")

mieszanina_filtered_max = mieszanina_filtered["width_hz"].max()
run_filtered_max = run_filtered["width_hz"].max()
print(mieszanina_filtered_max, run_filtered_max)

mieszanina_odrzucone_min = mieszanina_all["width_hz"].loc[mieszanina_all["width_hz"] > mieszanina_filtered_max].min()
print("Mieszanimieszanina odrzucone min", mieszanina_odrzucone_min)

substrat1_odrzucone_min = substrat1_all["width_hz"].loc[substrat1_all["width_hz"] > run_filtered_max].min()
substrat2_odrzucone_min = substrat2_all["width_hz"].loc[substrat2_all["width_hz"] > run_filtered_max].min()
print("Substrat1 odrzucone min", substrat1_odrzucone_min)
print("Substrat2 odrzucone min", substrat2_odrzucone_min)


2.11 3.08
Mieszanimieszanina odrzucone min 5.72
Substrat1 odrzucone min 3.32
Substrat2 odrzucone min 3.91
